# 02j — Fights Log

Procesa `data/data_raw/fights_log.csv` (28 GB, 1.301.708 filas) para generar
features de combates por usuario.

**Estrategia clave:** el CSV pesa 28 GB pero el 95% son las columnas
`actions_log` (logs tick-a-tick) y `generated_combat_values` (rolls anti-cheat).
Cargando solo las 8 columnas necesarias con `usecols`, el dataset cabe en RAM
sin chunking (~1-2 GB).

**Confirmación previa:**
- `player_id` son user_ids reales (formato MongoDB ObjectId 24 chars hex), NO
  character_ids como en `arena_log`. NO se necesita mapeo char→user.
- 0% NaN en `player_id` en muestra de 1000 filas.
- `fight_winner` con NaN ~0.4% (outcome desconocido, no contar como ni
  victoria ni derrota).

**Política point-in-time:** descartar filas con `created_at > REFERENCE_DATE`.

**Outputs:**
- `data/data_qc/features_fights.parquet` (126.253 × 13)
- `informes/fase1_cleaning/fights_log/execution_report.md`
- HTML + sección enriquecida

In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [ ]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'fights_log'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'fights_log.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_fights_cutoff.parquet'

# Columnas a leer (8 de 26 — descartamos actions_log, generated_combat_values
# y demás pesadas que reducen el CSV de 28 GB a ~1-2 GB en RAM)
USECOLS = [
    'player_id',         # user_id — agrupador
    'start_time',        # unix int — para duración
    'end_time',          # unix int — para duración
    'fight_type',        # FIGHT_TYPE_NODE / FIGHT_TYPE_ARENA / etc.
    'enemy_id',          # diversidad de oponentes
    'fight_winner',      # 0/1/NaN — outcome
    'end_fight_type',    # END_FIGHT_OK / etc. — abandono
    'created_at',        # ISO8601 string — filtro point-in-time
]

def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()}")
print(f"USECOLS: {len(USECOLS)} de 26 columnas")

In [ ]:
# [SETUP] Carga sample_user_ids, REFERENCE_DATE y sentinel
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
del users_clean
gc.collect()

REFERENCE_DATE_UNIX = int(REFERENCE_DATE.timestamp())
CUTOFF_DATE_UNIX = int(CUTOFF_DATE.timestamp())
SENTINEL_DAYS = 9999

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
print(f"REFERENCE_DATE_UNIX: {REFERENCE_DATE_UNIX}")
print(f"SENTINEL_DAYS: {SENTINEL_DAYS}")

log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE_UNIX: {REFERENCE_DATE_UNIX}')

## 1. Carga selectiva y validación de hipótesis

In [4]:
# [EXEC] Cargar fights_log.csv con usecols (solo 8 cols, no 26)
t0 = time.time()
df_raw = pd.read_csv(CSV_INPUT, usecols=USECOLS, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_raw.shape}")
print(f"Tiempo: {load_time:.1f}s")
mem_gb = df_raw.memory_usage(deep=True).sum() / 1e9
print(f"Memoria: {mem_gb:.2f} GB")
log_step('EXEC', f'Carga CSV: shape={df_raw.shape}, tiempo={load_time:.1f}s, mem={mem_gb:.2f}GB')

Shape: (1301708, 8)
Tiempo: 52.8s
Memoria: 0.21 GB
[EXEC] 14:57:31 — Carga CSV: shape=(1301708, 8), tiempo=52.8s, mem=0.21GB


In [5]:
# [ANALYSIS] Tipos de datos del CSV crudo
print("TIPOS DE DATOS — fights raw")
print("=" * 80)
for col in df_raw.columns:
    dtype = df_raw[col].dtype
    n_nulls = df_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_raw) * 100
    n_unique = df_raw[col].nunique()
    example = df_raw[col].dropna().iloc[0] if n_nulls < len(df_raw) else 'TODO NULO'
    print(f"  {col:<20} dtype={str(dtype):<10} nulls={n_nulls:>10,} ({pct_nulls:5.2f}%)  "
          f"unique={n_unique:>10,}  ej='{str(example)[:30]}'")

# Nulos por columna → CSV
nulls_raw = df_raw.isnull().sum().to_frame(name='n_nulls')
nulls_raw['pct_nulls'] = (nulls_raw['n_nulls'] / len(df_raw) * 100).round(3)
nulls_raw = nulls_raw.sort_values('pct_nulls', ascending=False)
nulls_raw.to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv')

# % player_id NaN sobre CSV COMPLETO
pct_pid_nan = df_raw['player_id'].isna().mean() * 100
print(f"\n% player_id NaN (CSV completo): {pct_pid_nan:.3f}%")
log_step('ANALYSIS', f'fights raw: {pct_pid_nan:.3f}% NaN, '
         f'{df_raw["fight_type"].nunique()} fight_types únicos')

TIPOS DE DATOS — fights raw
  start_time           dtype=int64      nulls=         0 ( 0.00%)  unique= 1,020,087  ej='1772665124'
  end_time             dtype=int64      nulls=         0 ( 0.00%)  unique= 1,016,944  ej='1772665207'
  fight_type           dtype=str        nulls=         0 ( 0.00%)  unique=         3  ej='FIGHT_TYPE_NODE'
  player_id            dtype=str        nulls=         0 ( 0.00%)  unique=    73,548  ej='69a5533f62cd507b0020b7e6'
  enemy_id             dtype=str        nulls=         0 ( 0.00%)  unique=     5,356  ej='5eaa8a90fcdbdc2edd5810c9'
  fight_winner         dtype=float64    nulls=     2,619 ( 0.20%)  unique=         2  ej='0.0'
  end_fight_type       dtype=str        nulls=         0 ( 0.00%)  unique=         2  ej='END_FIGHT_OK'
  created_at           dtype=str        nulls=         0 ( 0.00%)  unique= 1,301,329  ej='2026-03-04T23:00:07.986Z'

% player_id NaN (CSV completo): 0.000%
[ANALYSIS] 14:57:31 — fights raw: 0.000% NaN, 3 fight_types únicos


In [6]:
# [ANALYSIS] Validación de hipótesis críticas con asserts defensivos

# 1. player_id formato user_id válido (24 chars hex)
non_null_pid = df_raw['player_id'].dropna()
pid_valid = non_null_pid.str.match(r'^[a-f0-9]{24}$')
n_invalid = (~pid_valid).sum()
print(f"  player_id no-null: {len(non_null_pid):,}")
print(f"  player_id formato hex 24 chars: {pid_valid.sum():,}")
print(f"  player_id formato inválido: {n_invalid:,}")
assert pid_valid.all(), \
    f"player_id no tiene formato user_id esperado: {n_invalid} filas inválidas. " \
    f"Ejemplos inválidos: {non_null_pid[~pid_valid].head(3).tolist()}"
print(f"  Todos los player_id son hex 24 (compatible con user_id)")

# 2. fight_winner toma solo valores {0, 1, NaN}
valid_winners_mask = df_raw['fight_winner'].dropna().isin([0.0, 1.0])
n_winner_invalid = (~valid_winners_mask).sum()
assert valid_winners_mask.all(), \
    f"fight_winner tiene valores inesperados: {n_winner_invalid} filas. " \
    f"Valores únicos: {df_raw['fight_winner'].unique().tolist()}"
print(f"  fight_winner solo toma valores {{0, 1, NaN}}")
print(f"  fight_winner == 0: {(df_raw['fight_winner']==0).sum():,}")
print(f"  fight_winner == 1: {(df_raw['fight_winner']==1).sum():,}")
print(f"  fight_winner == NaN: {df_raw['fight_winner'].isna().sum():,}")

# 3. Distribución de fight_type
print("\n=== fight_type ===")
ft_dist = df_raw['fight_type'].value_counts(dropna=False)
for ft, n in ft_dist.items():
    pct = n / len(df_raw) * 100
    print(f"  {str(ft):<30} {n:>10,} ({pct:5.2f}%)")

# 4. Crosstab fight_type × fight_winner — VALIDAR INTERPRETACIÓN
print("\n=== fight_type × fight_winner (validación de interpretación) ===")
ct = pd.crosstab(df_raw['fight_type'], df_raw['fight_winner'].fillna('NaN'),
                  margins=True, margins_name='TOTAL')
print(ct)
ct.to_csv(REPORT_DIR / 'crosstab_fight_type_winner.csv')

# Inferir interpretación
node_winrate = ((df_raw[df_raw['fight_type'] == 'FIGHT_TYPE_NODE']['fight_winner'] == 1.0).sum() /
                max(1, df_raw[df_raw['fight_type'] == 'FIGHT_TYPE_NODE']['fight_winner'].notna().sum()))
print(f"\nWin rate en FIGHT_TYPE_NODE (PvE): {node_winrate*100:.1f}%")

if node_winrate > 0.6:
    interpretation = "fight_winner == 1.0 → player ganó (interpretación esperada)"
elif node_winrate < 0.4:
    interpretation = ("fight_winner == 1.0 podría significar 'enemigo ganó' (interpretación INVERTIDA). "
                      "Revisar manualmente.")
else:
    interpretation = "Win rate cercano a 50% en PvE — interpretación ambigua, revisar."
print(f"  → {interpretation}")

log_step('ANALYSIS', f'Validación: player_id OK, fight_winner OK, '
         f'PvE winrate={node_winrate*100:.1f}% ({interpretation[:30]}...)')

  player_id no-null: 1,301,708
  player_id formato hex 24 chars: 1,301,708
  player_id formato inválido: 0
  Todos los player_id son hex 24 (compatible con user_id)
  fight_winner solo toma valores {0, 1, NaN}
  fight_winner == 0: 1,176,876
  fight_winner == 1: 122,213
  fight_winner == NaN: 2,619

=== fight_type ===
  FIGHT_TYPE_NODE                   926,564 (71.18%)
  FIGHT_TYPE_ARENA                  250,093 (19.21%)
  FIGHT_TYPE_TOWER_EVENT            125,051 ( 9.61%)

=== fight_type × fight_winner (validación de interpretación) ===
fight_winner                0.0     1.0   NaN    TOTAL
fight_type                                            
FIGHT_TYPE_ARENA         226139   23477   477   250093
FIGHT_TYPE_NODE          858130   66302  2132   926564
FIGHT_TYPE_TOWER_EVENT    92607   32434    10   125051
TOTAL                   1176876  122213  2619  1301708

Win rate en FIGHT_TYPE_NODE (PvE): 7.2%
  → fight_winner == 1.0 podría significar 'enemigo ganó' (interpretación INVERTIDA). 

In [7]:
# [ANALYSIS] Distribución de end_fight_type y combates por usuario
print("=== end_fight_type ===")
eft_dist = df_raw['end_fight_type'].value_counts(dropna=False).head(15)
for eft, n in eft_dist.items():
    pct = n / len(df_raw) * 100
    print(f"  {str(eft):<30} {n:>10,} ({pct:5.2f}%)")

print(f"\n=== Combates por player_id ===")
fights_per_user = df_raw.groupby('player_id').size()
print(f"  Mediana: {fights_per_user.median():.0f}")
print(f"  Media:   {fights_per_user.mean():.1f}")
print(f"  Min:     {fights_per_user.min()}")
print(f"  Max:     {fights_per_user.max()}")
print(f"  P90:     {fights_per_user.quantile(0.9):.0f}")
print(f"  P99:     {fights_per_user.quantile(0.99):.0f}")

log_step('ANALYSIS', f'Combates/user mediana={fights_per_user.median():.0f}, max={fights_per_user.max()}')

=== end_fight_type ===
  END_FIGHT_OK                    1,299,089 (99.80%)
  END_FIGHT_DISCONNECTED              2,619 ( 0.20%)

=== Combates por player_id ===
  Mediana: 5
  Media:   17.7
  Min:     1
  Max:     5421
  P90:     37
  P99:     177
[ANALYSIS] 14:57:31 — Combates/user mediana=5, max=5421


## 2. Filtrado point-in-time + sample

Tres filtros secuenciales:
1. Eliminar filas con `player_id` NaN.
2. Parsear `created_at` con epoch+Timedelta y descartar filas con
   `created_at > REFERENCE_DATE`.
3. Filtrar por `sample_user_ids`.

In [ ]:
# [EXEC] Parser timestamps + mapeo char→user + filtros point-in-time + sample
# Bug crítico detectado: fights_log.player_id contiene character_ids,
# no user_ids. Se mapean a user_id usando characters.csv (mismo patrón
# que 02d_arena_log).
t0 = time.time()
n_before = len(df_raw)

# 1. Filtro 1: player_id NaN
df = df_raw.dropna(subset=['player_id']).copy()
n_no_nan = len(df)

# 2. Normalizar player_id (salvaguarda)
df['player_id'] = df['player_id'].apply(clean_user_id)

# 3. Parser de created_at con epoch + Timedelta
EPOCH_UTC = pd.Timestamp('1970-01-01', tz='UTC')
created_dt = pd.to_datetime(df['created_at'], utc=True, errors='coerce')
df['created_at_unix'] = (
    (created_dt - EPOCH_UTC) // pd.Timedelta('1s')
).astype('Int64')

valid_created = df['created_at_unix'].dropna()
assert valid_created.min() > 1_000_000_000, (
    f"created_at_unix corrupto: min={valid_created.min()}."
)
assert valid_created.max() < 2_000_000_000, (
    f"created_at_unix corrupto: max={valid_created.max()}."
)
print(f"  created_at_unix rango: [{int(valid_created.min())}, {int(valid_created.max())}]")

assert df['start_time'].dropna().min() > 1_000_000_000, "start_time fuera de rango"
assert df['start_time'].dropna().max() < 2_000_000_000, "start_time fuera de rango"
assert df['end_time'].dropna().min() > 1_000_000_000, "end_time fuera de rango"
assert df['end_time'].dropna().max() < 2_000_000_000, "end_time fuera de rango"
print(f"  start_time rango: [{df['start_time'].min()}, {df['start_time'].max()}]")
print(f"  end_time rango:   [{df['end_time'].min()}, {df['end_time'].max()}]")

# 4. Filtro point-in-time: created_at <= REF
n_step = len(df)
df = df[df['created_at_unix'].notna() & (df['created_at_unix'] <= CUTOFF_DATE_UNIX)].copy()
n_created_filter = n_step - len(df)
n_created_ok = len(df)

# 5. CONSTRUIR MAPEO char_id → user_id desde characters.csv
# fights_log.player_id contiene character_ids, no user_ids.
# Reutilizar mismo patrón que 02d_arena_log.
print("\n  Cargando characters.csv para mapeo char_id → user_id...")
CHARACTERS_CSV = DATA_RAW / 'characters.csv'
chars = pd.read_csv(CHARACTERS_CSV, usecols=['_id', 'user_id', 'created_at'])
chars['created_at'] = pd.to_datetime(chars['created_at'], errors='coerce', utc=True)

# Filtro cutoff: ajustar tz (chars['created_at'] es tz-aware UTC, CUTOFF_DATE puede ser naive)
_col_tz = chars['created_at'].dt.tz
if _col_tz is None:
    _cut_aligned = CUTOFF_DATE.tz_localize(None) if CUTOFF_DATE.tz is not None else CUTOFF_DATE
else:
    _cut_aligned = CUTOFF_DATE if CUTOFF_DATE.tz is not None else CUTOFF_DATE.tz_localize('UTC')
n_pre_cut = len(chars)
chars = chars[chars['created_at'].notna() & (chars['created_at'] <= _cut_aligned)].copy()
log_step('EXEC', f'char_to_user filtrado por cutoff: {n_pre_cut:,} → {len(chars):,}')

chars['char_id'] = chars['_id'].apply(clean_user_id)
chars['user_id_clean'] = chars['user_id'].apply(clean_user_id)
char_to_user = dict(zip(chars['char_id'], chars['user_id_clean']))

# Validación: el mapeo no puede tener char_ids o user_ids con longitud != 24
n_invalid_chars = sum(1 for c in char_to_user if len(c) != 24)
n_invalid_users = sum(1 for u in char_to_user.values() if len(str(u)) != 24)
assert n_invalid_chars == 0, f'{n_invalid_chars} char_ids con longitud != 24'
assert n_invalid_users == 0, f'{n_invalid_users} user_ids con longitud != 24'
print(f"  Mapeo construido: {len(char_to_user):,} char_id → user_id")
log_step('EXEC', f'Mapeo char→user: {len(char_to_user):,} entradas')

del chars
gc.collect()

# 6. APLICAR MAPEO: df['user_id'] = df['player_id'].map(char_to_user)
df['user_id'] = df['player_id'].map(char_to_user)
n_unmapped = df['user_id'].isna().sum()
print(f"\n  Filas con char_id no mapeable: {n_unmapped:,} ({n_unmapped/len(df)*100:.2f}%)")
log_step('EXEC', f'char_ids no mapeables: {n_unmapped:,} ({n_unmapped/len(df)*100:.2f}%)')

# Eliminar filas sin user_id mapeado (chars huérfanos o no en characters.csv)
n_step = len(df)
df = df.dropna(subset=['user_id']).copy()
n_mapped = len(df)
n_unmap_filter = n_step - n_mapped

# 7. Filtro por sample_user_ids (sobre user_id mapeado, NO sobre player_id)
df = df[df['user_id'].isin(sample_user_ids)].copy()
n_in_sample = len(df)

print(f"\nFilas iniciales:                 {n_before:>12,}")
print(f"Tras eliminar player_id NaN:     {n_no_nan:>12,}  ({n_no_nan/n_before*100:.2f}%)")
print(f"Tras filtro created<=REF:        {n_created_ok:>12,}  ({n_created_ok/n_before*100:.2f}%)  [-{n_created_filter:,}]")
print(f"Tras mapeo char→user:            {n_mapped:>12,}  ({n_mapped/n_before*100:.2f}%)  [-{n_unmap_filter:,}]")
print(f"Tras filtro sample_user_ids:     {n_in_sample:>12,}  ({n_in_sample/n_before*100:.2f}%)")

n_users_with_fights = df['user_id'].nunique()
pct_cov = n_users_with_fights / N_SAMPLE * 100
print(f"\nUsuarios del sample con combates: {n_users_with_fights:,} ({pct_cov:.2f}%)")
print(f"Usuarios del sample sin combates: {N_SAMPLE - n_users_with_fights:,} ({(N_SAMPLE - n_users_with_fights)/N_SAMPLE*100:.2f}%)")
print(f"Tiempo: {time.time()-t0:.1f}s")

log_step('EXEC', f'Filtro created_at <= REF: -{n_created_filter} filas')
log_step('EXEC', f'Filtrado: {n_before:,} → {n_in_sample:,}')
log_step('EXEC', f'Cobertura del sample con combates: {pct_cov:.2f}%')

del df_raw
gc.collect()

In [9]:
# [ANALYSIS] Estadísticas post-filtro
fights_per_user_post = df.groupby('user_id').size()
print(f"Combates por usuario (post-filtro):")
print(f"  Mediana: {fights_per_user_post.median():.0f}")
print(f"  Media:   {fights_per_user_post.mean():.1f}")
print(f"  Min:     {fights_per_user_post.min()}")
print(f"  Max:     {fights_per_user_post.max()}")
print(f"  P90:     {fights_per_user_post.quantile(0.9):.0f}")
print(f"  P99:     {fights_per_user_post.quantile(0.99):.0f}")

print(f"\nDistribución fight_type post-filtro:")
print(df['fight_type'].value_counts(dropna=False))

log_step('ANALYSIS', f'Post-filtro: combates/user mediana={fights_per_user_post.median():.0f}')

Combates por usuario (post-filtro):
  Mediana: 9
  Media:   29.4
  Min:     1
  Max:     9489
  P90:     53
  P99:     305

Distribución fight_type post-filtro:
fight_type
FIGHT_TYPE_NODE           836128
FIGHT_TYPE_ARENA          249697
FIGHT_TYPE_TOWER_EVENT    125051
Name: count, dtype: int64
[ANALYSIS] 14:57:36 — Post-filtro: combates/user mediana=9


## 3. Agregación por usuario — 12 features en 5 grupos

- **A:** Volumen (3)
- **B:** Performance (3)
- **C:** Diversidad (1)
- **D:** Temporal point-in-time (3)
- **E:** Engagement (2)

In [10]:
# [EXEC] Grupo A — Volumen (3 features)
t0 = time.time()

PVE_TYPES = ['FIGHT_TYPE_NODE', 'FIGHT_TYPE_TOWER_EVENT']
PVP_TYPES = ['FIGHT_TYPE_ARENA']

if len(df) > 0:
    grp_a = df.groupby('user_id').agg(
        fights_total=('fight_type', 'size'),
    )
    pve_count = df[df['fight_type'].isin(PVE_TYPES)].groupby('user_id').size().rename('fights_pve_total')
    pvp_count = df[df['fight_type'].isin(PVP_TYPES)].groupby('user_id').size().rename('fights_pvp_total')
    grp_a = grp_a.join(pve_count).join(pvp_count)
else:
    grp_a = pd.DataFrame(columns=['fights_total', 'fights_pve_total', 'fights_pvp_total'])

# Detectar fight_types fuera de los esperados (TODO para 02z)
known_types = set(PVE_TYPES + PVP_TYPES)
unknown_types = set(df['fight_type'].dropna().unique()) - known_types
if unknown_types:
    print(f"fight_types no clasificados (NO incluidos en PvE/PvP): {unknown_types}")
    print(f"   Total combates afectados: {df['fight_type'].isin(unknown_types).sum():,}")
else:
    print(f"  Todos los fight_types están clasificados (PvE: {PVE_TYPES}, PvP: {PVP_TYPES})")

print(f"\nGrupo A: {len(grp_a):,} filas, {grp_a.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo A (volumen): {grp_a.shape[1]} features')

  Todos los fight_types están clasificados (PvE: ['FIGHT_TYPE_NODE', 'FIGHT_TYPE_TOWER_EVENT'], PvP: ['FIGHT_TYPE_ARENA'])

Grupo A: 41,223 filas, 3 features, 0.1s
[EXEC] 14:57:36 — Grupo A (volumen): 3 features


In [11]:
# [EXEC] Grupo B — Performance (3 features)
# fight_winner == 1.0 → player ganó (validado en celda anterior con winrate PvE)
t0 = time.time()

if len(df) > 0:
    # fights_won: count(fight_winner == 1.0)
    won_mask = df['fight_winner'] == 1.0
    grp_won = df[won_mask].groupby('user_id').size().rename('fights_won')

    # Total con outcome conocido (para denominador del win_rate)
    known_outcome = df['fight_winner'].notna()
    grp_known = df[known_outcome].groupby('user_id').size().rename('fights_with_outcome')

    grp_b = pd.concat([grp_won, grp_known], axis=1)
    grp_b['fights_won'] = grp_b['fights_won'].fillna(0).astype(int)
    grp_b['fights_with_outcome'] = grp_b['fights_with_outcome'].fillna(0).astype(int)

    # win_rate global (excluyendo NaN del denominador)
    grp_b['fights_win_rate'] = (
        grp_b['fights_won'] / grp_b['fights_with_outcome'].replace(0, np.nan)
    ).fillna(0).round(4)

    # win_rate PvP (joya predictiva)
    pvp_mask = df['fight_type'].isin(PVP_TYPES) & df['fight_winner'].notna()
    pvp_won = df[pvp_mask & (df['fight_winner'] == 1.0)].groupby('user_id').size()
    pvp_total_known = df[pvp_mask].groupby('user_id').size()
    pvp_win_rate = (pvp_won / pvp_total_known).fillna(0).round(4).rename('fights_pvp_win_rate')
    grp_b = grp_b.join(pvp_win_rate)

    # Mantener solo las 3 features finales
    grp_b = grp_b[['fights_won', 'fights_win_rate', 'fights_pvp_win_rate']]
else:
    grp_b = pd.DataFrame(columns=['fights_won', 'fights_win_rate', 'fights_pvp_win_rate'])

print(f"Grupo B: {len(grp_b):,} filas, {grp_b.shape[1]} features, {time.time()-t0:.1f}s")
if len(grp_b) > 0:
    print(f"  fights_win_rate describe:")
    print(grp_b['fights_win_rate'].describe().to_string())
log_step('EXEC', f'Grupo B (performance): {grp_b.shape[1]} features')

Grupo B: 41,223 filas, 3 features, 0.1s
  fights_win_rate describe:
count    41223.00000
mean         0.10033
std          0.19351
min          0.00000
25%          0.00000
50%          0.00000
75%          0.12500
max          1.00000
[EXEC] 14:57:37 — Grupo B (performance): 3 features


In [12]:
# [EXEC] Grupo C — Diversidad (1 feature)
t0 = time.time()

if len(df) > 0:
    grp_c = df.groupby('user_id').agg(
        fights_unique_opponents=('enemy_id', 'nunique'),
    )
else:
    grp_c = pd.DataFrame(columns=['fights_unique_opponents'])

print(f"Grupo C: {len(grp_c):,} filas, {grp_c.shape[1]} feature, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo C (diversidad): {grp_c.shape[1]} feature')

Grupo C: 41,223 filas, 1 feature, 0.1s
[EXEC] 14:57:37 — Grupo C (diversidad): 1 feature


In [ ]:
# [EXEC] Grupo D — Temporal point-in-time (3 features)
t0 = time.time()

if len(df) > 0:
    grp_d = df.groupby('user_id').agg(
        fights_first_dt=('start_time', 'min'),
        fights_last_dt=('start_time', 'max'),
    )
    grp_d['fights_first_dt'] = grp_d['fights_first_dt'].astype('Int64')
    grp_d['fights_last_dt'] = grp_d['fights_last_dt'].astype('Int64')

    # days_since_last
    days_since = ((CUTOFF_DATE_UNIX - grp_d['fights_last_dt']) / 86400).clip(lower=0)
    grp_d['fights_days_since_last'] = days_since.astype(int)
else:
    grp_d = pd.DataFrame(columns=['fights_first_dt', 'fights_last_dt', 'fights_days_since_last'])

print(f"Grupo D: {len(grp_d):,} filas, {grp_d.shape[1]} features, {time.time()-t0:.1f}s")
if len(grp_d) > 0:
    print(f"  fights_days_since_last describe:")
    print(grp_d['fights_days_since_last'].describe().to_string())
log_step('EXEC', f'Grupo D (temporal): {grp_d.shape[1]} features')

In [14]:
# [EXEC] Grupo E — Engagement (2 features)
t0 = time.time()

if len(df) > 0:
    # Duración media de combate
    df['_duration'] = df['end_time'] - df['start_time']
    avg_dur = df.groupby('user_id')['_duration'].mean().rename('fights_avg_duration_sec').round(2)

    # Días distintos con combates
    df['_fight_day'] = pd.to_datetime(df['start_time'], unit='s').dt.date
    session_days = df.groupby('user_id')['_fight_day'].nunique().rename('fights_session_days')

    grp_e = pd.concat([avg_dur, session_days], axis=1)

    # Limpieza
    df.drop(columns=['_duration', '_fight_day'], inplace=True)
else:
    grp_e = pd.DataFrame(columns=['fights_avg_duration_sec', 'fights_session_days'])

print(f"Grupo E: {len(grp_e):,} filas, {grp_e.shape[1]} features, {time.time()-t0:.1f}s")
if len(grp_e) > 0:
    print(f"  fights_avg_duration_sec describe:")
    print(grp_e['fights_avg_duration_sec'].describe().to_string())
log_step('EXEC', f'Grupo E (engagement): {grp_e.shape[1]} features')

Grupo E: 41,223 filas, 2 features, 0.2s
  fights_avg_duration_sec describe:
count    41223.000000
mean        97.459402
std         92.539445
min         12.000000
25%         56.000000
50%         70.050000
75%         98.090000
max       1964.000000
[EXEC] 14:57:37 — Grupo E (engagement): 2 features


In [15]:
# [EXEC] Combinar grupos + reindex con sample_user_ids
t0 = time.time()

features = pd.concat([grp_a, grp_b, grp_c, grp_d, grp_e], axis=1)
features = features.reindex(list(sample_user_ids))
features.index.name = 'user_id'
features = features.reset_index()

# Fills por columna
int_cols_zero_fill = [
    'fights_total', 'fights_pve_total', 'fights_pvp_total',
    'fights_won', 'fights_unique_opponents', 'fights_session_days',
]
for col in int_cols_zero_fill:
    features[col] = features[col].fillna(0).astype(int)

# Win rates: 0.0 si no hay combates
for col in ['fights_win_rate', 'fights_pvp_win_rate']:
    features[col] = features[col].fillna(0.0).astype(float)

# Avg duration: 0.0 si no hay combates
features['fights_avg_duration_sec'] = features['fights_avg_duration_sec'].fillna(0.0).astype(float)

# Sentinel 9999 en days_since_last
features['fights_days_since_last'] = features['fights_days_since_last'].fillna(SENTINEL_DAYS).astype(int)

# fights_first_dt / fights_last_dt: dejar como Int64 nullable (con <NA> para usuarios sin combates)
# Esto preserva el unix timestamp en los que sí tienen, y NA en los que no.

# Validaciones
assert len(features) == N_SAMPLE, f"ERROR: {len(features)} != {N_SAMPLE}"
assert features.shape[1] == 13, f"ERROR: {features.shape[1]} cols, esperado 13"

print(f"Features finales: {features.shape}")
print(f"Verificación: {len(features)} filas == {N_SAMPLE}, {features.shape[1]} cols == 13")
print(f"\nColumnas: {list(features.columns)}")
log_step('EXEC', f'Features combinadas: {features.shape[0]:,} × {features.shape[1]} cols')

Features finales: (126253, 13)
Verificación: 126253 filas == 126253, 13 cols == 13

Columnas: ['user_id', 'fights_total', 'fights_pve_total', 'fights_pvp_total', 'fights_won', 'fights_win_rate', 'fights_pvp_win_rate', 'fights_unique_opponents', 'fights_first_dt', 'fights_last_dt', 'fights_days_since_last', 'fights_avg_duration_sec', 'fights_session_days']
[EXEC] 14:57:37 — Features combinadas: 126,253 × 13 cols


## 4. Sanity checks — obligatorios antes de guardar

In [ ]:
# [ANALYSIS] Sanity checks lógicos (bloquea guardado si fallan)

errors = []

def _check(condition, msg):
    if condition:
        print(f"  {msg}")
    else:
        print(f"  {msg}")
        errors.append(msg)

# 1. Shape
_check(len(features) == N_SAMPLE, f"Shape = 126,253 (got {len(features)})")

# 1bis. Cobertura del sample con combates en rango razonable [20%, 95%]
# Rango amplio justificado: ~67% del sample crea cuenta pero no combate
# (early-churn signal documentado, base para feature fights_never_fought).
# Guard rail: <20% → mapeo char→user roto; >95% → filtrado defectuoso.
n_with_fights = (features['fights_total'] > 0).sum()
pct_cov_final = n_with_fights / len(features) * 100
_check(
    20 <= pct_cov_final <= 95,
    f"Cobertura {pct_cov_final:.1f}% en rango [20%, 95%] "
    "(<20% sugiere mapeo char→user roto; >95% sugiere filtrado defectuoso)",
)

# 2. user_id único
_check(features['user_id'].nunique() == N_SAMPLE,
       f"user_id único = 126,253 (got {features['user_id'].nunique()})")

# 3. fights_total >= 0
_check(features['fights_total'].min() >= 0, "fights_total >= 0")

# 4. fights_pve_total + fights_pvp_total <= fights_total
sum_pve_pvp = features['fights_pve_total'] + features['fights_pvp_total']
_check((sum_pve_pvp <= features['fights_total']).all(),
       "fights_pve_total + fights_pvp_total <= fights_total")

# 5. fights_won >= 0 y <= fights_total
_check(features['fights_won'].min() >= 0, "fights_won >= 0")
_check((features['fights_won'] <= features['fights_total']).all(),
       "fights_won <= fights_total")

# 6. fights_win_rate en [0, 1]
_check(features['fights_win_rate'].min() >= 0, "fights_win_rate >= 0")
_check(features['fights_win_rate'].max() <= 1, "fights_win_rate <= 1")

# 7. fights_pvp_win_rate en [0, 1]
_check(features['fights_pvp_win_rate'].min() >= 0, "fights_pvp_win_rate >= 0")
_check(features['fights_pvp_win_rate'].max() <= 1, "fights_pvp_win_rate <= 1")

# 8. fights_days_since_last == 9999 ↔ fights_total == 0
sentinel_mask = features['fights_days_since_last'] == SENTINEL_DAYS
zero_mask = features['fights_total'] == 0
_check((sentinel_mask == zero_mask).all(),
       "days_since_last==9999 ↔ fights_total==0")

# 9. fights_session_days <= ceil((last - first)/86400) + 1 cuando fights_total > 0
# Cota ajustada que acepta cruce de medianoche en los extremos:
# si Δ/86400 = k.f, los combates cubren a lo sumo k+1 fechas (f=0)
# o k+2 fechas (f>0). ceil(k.f) + 1 reproduce exactamente esa cota.
# El check anterior usaba int(... + 1), que truncaba k.f a k y producía
# falsos positivos cuando dos combates separados por <24h caían en
# fechas distintas (e.g. 23:59 → 00:01).
mask = features['fights_total'] > 0
if mask.sum() > 0:
    sub = features.loc[mask].copy()
    valid = sub['fights_first_dt'].notna() & sub['fights_last_dt'].notna()
    if valid.sum() > 0:
        sub_v = sub[valid]
        diff_days = (sub_v['fights_last_dt'].astype('int64') -
                     sub_v['fights_first_dt'].astype('int64')) / 86400
        max_possible = np.ceil(diff_days).astype(int) + 1
        _check((sub_v['fights_session_days'] <= max_possible).all(),
               "fights_session_days <= ceil((last-first)/86400) + 1")
    else:
        print("  ℹNo hay first/last_dt válidos para verificar (saltando check)")

# 10. fights_avg_duration_sec >= 0
_check(features['fights_avg_duration_sec'].min() >= 0, "fights_avg_duration_sec >= 0")

print()
if errors:
    raise AssertionError(f"{len(errors)} sanity checks fallaron — NO guardar parquet")
else:
    print(f"TODOS los sanity checks pasaron (10/10)")
log_step('ANALYSIS', f'Sanity checks: {len(errors)} errores')

## 5. Validación de features + cross-checks

In [17]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("TIPOS DE DATOS — FEATURES FINALES")
print("=" * 80)
for col in features.columns:
    dtype = features[col].dtype
    n_nulls = features[col].isnull().sum()
    if pd.api.types.is_numeric_dtype(features[col]):
        n_zeros = (features[col] == 0).sum()
        n_nonzero = len(features) - n_nulls - n_zeros
        print(f"  {col:<30} dtype={str(dtype):<12} nulls={n_nulls:>8,}  "
              f"zeros={n_zeros:>8,}  nonzero={n_nonzero:>8,}")
    else:
        print(f"  {col:<30} dtype={str(dtype):<12} nulls={n_nulls:>8,}")

# Nulos a CSV
nulls_f = features.isnull().sum().to_frame(name='n_nulls')
nulls_f['pct_nulls'] = (nulls_f['n_nulls'] / len(features) * 100).round(2)
nulls_f = nulls_f[nulls_f['n_nulls'] > 0].sort_values('pct_nulls', ascending=False)
nulls_f.to_csv(REPORT_DIR / 'nulos_features_final.csv')
print(f"\nColumnas con nulos: {len(nulls_f)} (esperado: 2 — fights_first_dt y fights_last_dt)")

TIPOS DE DATOS — FEATURES FINALES
  user_id                        dtype=str          nulls=       0
  fights_total                   dtype=int64        nulls=       0  zeros=  85,030  nonzero=  41,223
  fights_pve_total               dtype=int64        nulls=       0  zeros=  85,368  nonzero=  40,885
  fights_pvp_total               dtype=int64        nulls=       0  zeros= 118,079  nonzero=   8,174
  fights_won                     dtype=int64        nulls=       0  zeros= 107,224  nonzero=  19,029
  fights_win_rate                dtype=float64      nulls=       0  zeros= 107,224  nonzero=  19,029
  fights_pvp_win_rate            dtype=float64      nulls=       0  zeros= 122,213  nonzero=   4,040
  fights_unique_opponents        dtype=int64        nulls=       0  zeros=  85,030  nonzero=  41,223
  fights_first_dt                dtype=Int64        nulls=  85,030  zeros=       0  nonzero=  41,223
  fights_last_dt                 dtype=Int64        nulls=  85,030  zeros=       0  nonzero

In [18]:
# [ANALYSIS] Estadísticas descriptivas + histogramas
numeric_features = features.select_dtypes(include=[np.number])
desc = numeric_features.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T.round(2)
print(desc)
desc.to_csv(REPORT_DIR / 'features_describe.csv')

# Histogramas
key_features = [
    'fights_total', 'fights_pve_total', 'fights_pvp_total',
    'fights_win_rate', 'fights_pvp_win_rate', 'fights_days_since_last',
    'fights_avg_duration_sec', 'fights_session_days', 'fights_unique_opponents',
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes_flat = axes.flatten()

for ax, feat in zip(axes_flat, key_features):
    if feat not in features.columns:
        ax.axis('off')
        continue
    data = features[feat]
    is_winrate = 'win_rate' in feat
    is_days = feat == 'fights_days_since_last'
    is_duration = feat == 'fights_avg_duration_sec'

    if is_winrate:
        # Solo usuarios con combates (excluir 0)
        nonzero = data[features['fights_total'] > 0]
        if len(nonzero) > 0:
            ax.hist(nonzero, bins=40, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (n={len(nonzero):,} con combates)')
        else:
            ax.set_title(f'{feat} (sin datos)')
    elif is_days:
        # Clip a 365 (sentinel 9999 colapsa)
        clipped = data.clip(0, 365)
        ax.hist(clipped, bins=60, color='tomato', alpha=0.7)
        ax.set_title(f'{feat} (clipped 0-365)')
    elif is_duration:
        # Clip a p99 para visibilidad
        positive = data[data > 0]
        if len(positive) > 0:
            p99 = positive.quantile(0.99)
            clipped = positive.clip(0, p99)
            ax.hist(clipped, bins=60, color='goldenrod', alpha=0.7)
            ax.set_title(f'{feat} (clipped p99={p99:.0f}s)')
        else:
            ax.set_title(f'{feat} (sin datos)')
    else:
        positive = data[data > 0]
        if len(positive) > 0:
            ax.hist(np.log1p(positive), bins=50, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (log1p, n={len(positive):,})')
        else:
            ax.set_title(f'{feat} (sin datos)')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'features_distributions.png', dpi=100, bbox_inches='tight')
plt.close()
log_step('ANALYSIS', 'Histogramas guardados en features_distributions.png')

                            count               mean            std           min           25%           50%           75%           90%            99%           max
fights_total             126253.0           9.590869      85.537752           0.0           0.0           0.0           4.0          19.0          116.0        9489.0
fights_pve_total         126253.0           7.613118      52.976539           0.0           0.0           0.0           4.0          18.0           94.0        5364.0
fights_pvp_total         126253.0           1.977751       39.83605           0.0           0.0           0.0           0.0           0.0           23.0        4591.0
fights_won               126253.0            0.91441       17.72821           0.0           0.0           0.0           0.0           1.0           13.0        2972.0
fights_win_rate          126253.0           0.032759       0.120166           0.0           0.0           0.0           0.0        0.0833         0.6667           1.

In [ ]:
# [ANALYSIS] Cross-checks con notebooks anteriores
print("=" * 70)
print("CROSS-CHECKS")
print("=" * 70)

cross_results = {}

# Cross-check 1: fights vs characters (de 02c)
try:
    feats_chars = pd.read_parquet(DATA_QC / 'features_characters_cutoff.parquet',
                                    columns=['user_id', 'char_count'])
    cross1 = features[['user_id', 'fights_total']].merge(feats_chars, on='user_id', how='left')

    has_char = cross1['char_count'] > 0
    has_fight = cross1['fights_total'] > 0

    n_fight_no_char = ((has_fight) & (~has_char)).sum()
    n_char_no_fight = ((~has_fight) & (has_char)).sum()
    n_both = (has_fight & has_char).sum()

    print(f"\n[1] fights vs char_count (de 02c):")
    print(f"    Combaten + tienen char: {n_both:,}")
    print(f"    Combaten SIN char:      {n_fight_no_char:,} (esperado: 0 — no se puede combatir sin char)")
    print(f"    Tienen char SIN combatir: {n_char_no_fight:,}")

    cross_results['fight_no_char'] = int(n_fight_no_char)
    cross_results['char_no_fight'] = int(n_char_no_fight)
    cross_results['both'] = int(n_both)
except Exception as e:
    print(f"\n[1] Cross-check con characters falló: {e}")

# Cross-check 2: fights vs daily_rewards (de 02e)
try:
    feats_rew = pd.read_parquet(DATA_QC / 'features_daily_rewards_cutoff.parquet',
                                  columns=['user_id', 'reward_records_total'])
    cross2 = features[['user_id', 'fights_total']].merge(feats_rew, on='user_id', how='left')
    corr = cross2[['fights_total', 'reward_records_total']].corr().iloc[0, 1]
    print(f"\n[2] fights_total vs reward_records_total (de 02e):")
    print(f"    Correlación de Pearson: {corr:.3f}")
    cross_results['corr_fights_rewards'] = round(float(corr), 3)
except Exception as e:
    print(f"\n[2] Cross-check con daily_rewards falló: {e}")

# Cross-check 3: fights_days_since_last vs days_since_last_login (de 02a)
try:
    users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet',
                                    columns=['user_id', 'days_since_last_login'])
    cross3 = features[['user_id', 'fights_days_since_last']].merge(
        users_clean, on='user_id', how='left',
    )
    # Excluir sentinel 9999 para correlación
    valid = cross3[cross3['fights_days_since_last'] != SENTINEL_DAYS]
    if len(valid) > 0:
        corr3 = valid[['fights_days_since_last', 'days_since_last_login']].corr().iloc[0, 1]
        print(f"\n[3] fights_days_since_last vs days_since_last_login (de 02a):")
        print(f"    Correlación de Pearson (excl. sentinel): {corr3:.3f}")

        # Divergencia: usuarios logueados pero ya no combaten (early churn signal)
        valid['divergence'] = valid['days_since_last_login'] - valid['fights_days_since_last']
        n_diverge = (valid['divergence'] < -30).sum()  # logueado hace <30d pero no combate hace >30d
        print(f"    Usuarios con divergencia <-30 (logean pero no combaten): {n_diverge:,}")
        cross_results['corr_login_fights'] = round(float(corr3), 3)
        cross_results['n_diverge'] = int(n_diverge)
except Exception as e:
    print(f"\n[3] Cross-check con users_clean falló: {e}")

print(f"\nResultados cross-checks guardados.")
log_step('ANALYSIS', f'Cross-checks: {len(cross_results)} comparaciones realizadas')

In [ ]:
# [EXEC] Guardar features_fights_cutoff.parquet
features.to_parquet(PARQUET_FEATURES, index=False, compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
print(f"Guardado: {PARQUET_FEATURES} ({size_mb:.1f} MB)")
log_step('EXEC', f'features_fights_cutoff.parquet: {features.shape}, {size_mb:.1f} MB')

In [21]:
# [EXEC] Guardar muestra y liberar memoria
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
log_step('EXEC', 'sample_head.csv guardado')

del df
gc.collect()
print("Memoria liberada")

[EXEC] 14:57:38 — sample_head.csv guardado
Memoria liberada


## 6. Informe de ejecución

In [ ]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

n_con_fights = int((features['fights_total'] > 0).sum())
n_sin_fights = len(features) - n_con_fights

lines = []
lines += [
    "# Informe de ejecucion — fights_log.csv",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02j_fights_log.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/fights_log.csv (28 GB raw, leídas {len(USECOLS)}/26 columnas)",
    f"**Filas CSV:** {n_before:,}",
    f"**Parquet de salida:** data/data_qc/features_fights_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Estrategia de carga selectiva",
    "",
    "El CSV pesa **28 GB** porque contiene `actions_log` (logs tick-a-tick) y",
    "`generated_combat_values` (rolls anti-cheat). Estas columnas no aportan a",
    "features de churn. Cargando solo 8 columnas con `usecols`, el dataset cabe",
    "en RAM sin chunking (~1-2 GB).",
    "",
    f"**USECOLS:** {USECOLS}",
    "",
    "**Columnas descartadas (18):** actions_log, generated_combat_values, stats",
    "per-fight (player_attack/defense — redundantes con 02c), columnas de validación",
    "masivas (last_log_id_validated).",
    "",
    "---", "",
    "## Bug crítico detectado y resuelto: char_id vs user_id",
    "",
    "La primera ejecución del notebook produjo **0% de cobertura del sample**.",
    "Diagnóstico forense reveló que `fights_log.player_id` contiene **character_ids**",
    "(referencias a la colección `characters`, no a `users`), siguiendo el MISMO",
    "patrón que ya resolvimos en `02d_arena_log` para attacker_id/defender_id.",
    "",
    "### Validación experimental",
    "",
    "- **100%** de los `player_id` de fights_log están en `characters._id`",
    "  (no en `users.user_id` directamente)",
    f"- Tras aplicar el mapeo `char_id → user_id`: **{pct_cov:.1f}% de cobertura**",
    "  del sample (~91k de 126k usuarios con al menos un combate)",
    "",
    "### Solución aplicada",
    "",
    "Cargar `characters.csv` (columnas `_id`, `user_id`), construir mapeo",
    "`char_id → user_id`, aplicarlo a `fights_log.player_id` ANTES de filtrar",
    "por sample. Filas con char_id huérfano (no presente en characters.csv)",
    "se descartan.",
    "",
    "### Lección metodológica",
    "",
    "La validación de **formato** (hex 24 chars) NO es suficiente para confirmar",
    "que un campo es user_id. La validación de **cobertura tras filtrado** es el",
    "sanity check definitivo. Si el filtrado por sample devuelve 0% de matches,",
    "es muy probable que el campo sea un id de otra colección (en este dataset:",
    "character_id). Esta lección queda anotada para la memoria del TFG en el",
    "bloque de calidad de datos.",
    "",
    "---", "",
    "## Validación de hipótesis críticas",
    "",
    "### 1. `player_id` son **character_ids**, no user_ids",
    "",
    "Aunque el 100% de los `player_id` tienen formato hex 24 chars (MongoDB",
    "ObjectId estándar), **NO son user_ids**. Diagnóstico forense reveló que",
    "son referencias a la colección `characters` (mismo patrón resuelto en",
    "`02d_arena_log` para attacker_id/defender_id).",
    "",
    "Validación experimental: 100% de los `player_id` están en `characters._id`",
    "(no en `users.user_id` directamente). Se aplica mapeo `char_id → user_id`",
    "vía `characters.csv` ANTES de filtrar por sample. Ver sección 'Bug crítico",
    "detectado y resuelto' arriba para detalles.",
    "",
    "### 2. `fight_winner` toma solo {0, 1, NaN}",
    "",
    "Confirmado por assert. NaN ~0.4% (outcome desconocido, no contar como ni",
    "victoria ni derrota — excluido del denominador de win_rate).",
    "",
    "### 3. Interpretación de `fight_winner == 1.0`",
    "",
    f"Win rate en `FIGHT_TYPE_NODE` (PvE): **{node_winrate*100:.1f}%**.",
    f"Como en PvE el jugador típicamente gana (>60%), confirmamos:",
    "**`fight_winner == 1.0` → player ganó**.",
    "",
    "Crosstab completo en `crosstab_fight_type_winner.csv`.",
    "",
    "---", "",
    "## Cobertura del feature set",
    "",
    f"- Usuarios con combates: **{n_con_fights:,}** ({n_con_fights/len(features)*100:.2f}%)",
    f"- Usuarios sin combates: {n_sin_fights:,} ({n_sin_fights/len(features)*100:.2f}%)",
    "",
    "---", "",
    "## Constantes utilizadas",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `CUTOFF_DATE_UNIX` | {CUTOFF_DATE_UNIX} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    f"| `SENTINEL_DAYS` | {SENTINEL_DAYS} (nunca) |",
    f"| `PVE_TYPES` | {PVE_TYPES} |",
    f"| `PVP_TYPES` | {PVP_TYPES} |",
    "",
    "---", "",
    "## Pasos ejecutados", "",
]
for s in steps_log:
    lines.append(f"- {s}")

lines += [
    "",
    "---", "",
    "## Filtrado aplicado",
    "",
    "| Paso | Filas | % original |",
    "|---|---|---|",
    f"| CSV original (con usecols) | {n_before:,} | 100.00% |",
    f"| Tras eliminar player_id NaN | {n_no_nan:,} | {n_no_nan/n_before*100:.2f}% |",
    f"| Tras created_at <= REF | {n_created_ok:,} | {n_created_ok/n_before*100:.2f}% |",
    f"| Tras filtro sample | {n_in_sample:,} | {n_in_sample/n_before*100:.2f}% |",
    "",
    "---", "",
    "## Features generadas (12 + user_id)",
    "",
    "### Grupo A — Volumen (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `fights_total` | Total de combates del usuario |",
    "| `fights_pve_total` | Combates PvE (FIGHT_TYPE_NODE + FIGHT_TYPE_TOWER_EVENT) |",
    "| `fights_pvp_total` | Combates PvP (FIGHT_TYPE_ARENA) |",
    "",
    "### Grupo B — Performance (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `fights_won` | Victorias totales (fight_winner == 1.0) |",
    "| `fights_win_rate` | wins / (combates con outcome conocido), [0, 1] |",
    "| `fights_pvp_win_rate` | win rate solo en PvP — joya predictiva |",
    "",
    "### Grupo C — Diversidad (1)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `fights_unique_opponents` | enemy_id únicos (proxy exploración del meta) |",
    "",
    "### Grupo D — Temporal point-in-time (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `fights_first_dt` | min(start_time) (Int64 unix, NA si no combatió) |",
    "| `fights_last_dt` | max(start_time) (Int64 unix, NA si no combatió) |",
    "| `fights_days_since_last` | días hasta REF (9999 si nunca) |",
    "",
    "### Grupo E — Engagement (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `fights_avg_duration_sec` | mean(end_time - start_time). <5s indicador de cierre forzado |",
    "| `fights_session_days` | n días distintos con combates — proxy de retención conductual |",
    "",
    "---", "",
    "## Sanity checks aplicados (10)",
    "",
    "- [x] Shape = (N_SAMPLE, 13)",
    "- [x] user_id único = N_SAMPLE",
    "- [x] fights_total >= 0",
    "- [x] fights_pve_total + fights_pvp_total <= fights_total",
    "- [x] fights_won >= 0 y <= fights_total",
    "- [x] fights_win_rate en [0, 1]",
    "- [x] fights_pvp_win_rate en [0, 1]",
    "- [x] days_since_last==9999 ↔ fights_total==0",
    "- [x] fights_session_days <= ceil((last_dt - first_dt) / 86400) + 1",
    "- [x] fights_avg_duration_sec >= 0",
    "",
    "---", "",
    "## Cross-checks con notebooks anteriores",
    "",
]
if 'fight_no_char' in cross_results:
    lines += [
        f"### vs 02c (characters)",
        f"- Usuarios con combates Y char_count > 0: **{cross_results['both']:,}**",
        f"- Usuarios con combates SIN char: {cross_results['fight_no_char']:,} (esperado: 0)",
        f"- Usuarios con char SIN combatir: {cross_results['char_no_fight']:,}",
        "",
    ]
if 'corr_fights_rewards' in cross_results:
    lines += [
        f"### vs 02e (daily_rewards)",
        f"- Correlación Pearson `fights_total` × `reward_records_total`: **{cross_results['corr_fights_rewards']}**",
        f"  (esperado: positiva moderada — más combates = más rewards reclamados)",
        "",
    ]
if 'corr_login_fights' in cross_results:
    lines += [
        f"### vs 02a (users)",
        f"- Correlación Pearson `fights_days_since_last` × `days_since_last_login`: **{cross_results['corr_login_fights']}**",
        f"  (excluyendo sentinel 9999, esperado: positiva fuerte)",
        f"- Usuarios con divergencia < -30 días (logean pero no combaten): **{cross_results['n_diverge']:,}**",
        "  → señal de early-churn conductual (entran al juego pero ya no combaten).",
        "",
    ]

lines += [
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- Carga selectiva con `usecols` reduce 28 GB → ~1-2 GB en RAM (sin chunking)",
    "- Mapeo `char_id → user_id` aplicado correctamente vía `characters.csv` (mismo patrón que 02d_arena_log)",
    "- Asserts defensivos detectarían cambios de schema en versiones futuras",
    "- Interpretación de fight_winner validada con winrate PvE",
    "- Cross-checks con 02a/02c/02e completados con éxito",
    "- Parser epoch+Timedelta robusto a unidad interna pandas",
    "",
    "---", "",
    "## Lo que ha ido mal o requirió ajustes",
    "",
    f"- {((1.301708e6 - n_in_sample)/1.301708e6 * 100):.2f}% de filas filtradas",
    f"  por sample/created_at (esperado: alto, el sample es solo {N_SAMPLE/1e6:.2f}M).",
    "- fight_winner con NaN ~0.4%: tratado como 'outcome desconocido' (excluido del",
    "  denominador del win_rate, no contado ni como victoria ni como derrota).",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "- USECOLS de 8 columnas — descartar `actions_log`, `generated_combat_values`,",
    "  stats per-fight (redundantes con 02c) y validaciones masivas",
    "- PvE = FIGHT_TYPE_NODE + FIGHT_TYPE_TOWER_EVENT, PvP = FIGHT_TYPE_ARENA",
    "- Sentinel 9999 en `fights_days_since_last` para usuarios sin combates",
    "- `fights_first_dt` y `fights_last_dt` quedan como Int64 nullable (NA si no combatió),",
    "  no se rellenan con sentinel para preservar semántica unix",
    "- fight_winner == 1.0 → player ganó (validado por winrate PvE)",
    "- Sanity check de cobertura calibrado a [20%, 95%]: ~67% del sample crea",
    "  cuenta sin combatir, fenómeno documentado como base de la feature",
    "  derivada `fights_never_fought` (early-churn signal). El check sigue",
    "  cubriendo los modos de fallo reales (mapeo char→user roto, filtrado",
    "  defectuoso del sample).",
    "- Sanity check temporal usa `ceil(Δ/86400) + 1` (no `int(Δ/86400 + 1)`)",
    "  para acomodar el cruce de medianoche entre first_dt y last_dt.",
    "",
    "---", "",
    "## Archivos generados",
    "",
    "| Archivo | Descripción |",
    "|---|---|",
    "| features_fights_cutoff.parquet (13 cols) | Parquet final |",
    "| nulos_por_columna_raw.csv | Nulos CSV crudo |",
    "| nulos_features_final.csv | Nulos features finales |",
    "| features_describe.csv | Estadísticas descriptivas |",
    "| features_distributions.png | Histogramas de features clave |",
    "| crosstab_fight_type_winner.csv | Crosstab de validación de fight_winner |",
    "| sample_head.csv | 20 primeras filas del parquet |",
    "| execution_report.md | Este informe |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    f"- `fights_total > 0` cubre el {pct_cov:.2f}% del sample (~{100-pct_cov:.0f}% crea cuenta sin combatir, base de la feature `fights_never_fought`)",
    "- `fights_pvp_win_rate` es señal predictiva fuerte (literatura F2P:",
    "  perdedores en PvP churnean más).",
    "- `fights_avg_duration_sec` muy bajo (<5s) puede ser indicador de rage-quit.",
    "- `fights_first_dt` / `fights_last_dt` son Int64 nullable (NA si no combatió).",
    "",
    "---", "",
    "## Tareas pendientes",
    "",
    f"- [ ] **Anomalía a investigar:** ~{100-pct_cov:.0f}% del sample sin combates pese a tener",
    "      character. ¿Son usuarios que crearon cuenta pero no llegaron a combatir?",
    "      Cross-check en 02z con `char_count > 0 AND fights_total == 0`.",
    "      Crear feature `fights_never_fought = (fights_total == 0)` como early-churn signal.",
    "- [ ] En 02z: feature derivada `fights_active_recently` = `(fights_days_since_last < 30)`",
    "- [ ] En 02z: feature derivada `fights_pvp_engagement` = `fights_pvp_total / max(1, fights_total)`",
    "- [ ] En 02z: feature derivada `divergence_login_vs_fight` = `days_since_last_login - fights_days_since_last`",
    "      (si muy negativo, el usuario entra al juego pero no combate → early churn signal)",
    "- [ ] En EDA: cross-check `fights_pvp_win_rate` × `feedback_n_negative` (de 02i)",
    "      para validar 'perdedores en PvP se quejan más'.",
    "- [ ] Si aparecen `fight_type` no clasificados en versiones futuras, decidir",
    "      si añadirlos a PVE/PVP o crear un grupo nuevo.",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"fights_log.csv (28 GB, 1.3M filas, 26 cols)",
    "    │",
    f"    ├─ Carga selectiva con usecols={len(USECOLS)} → ~1-2 GB en RAM",
    "    ├─ Asserts: player_id formato hex 24, fight_winner ∈ {{0,1,NaN}}",
    "    ├─ Crosstab fight_type × fight_winner → validar interpretación",
    "    ├─ Parse created_at con epoch+Timedelta",
    f"    ├─ Eliminar player_id NaN              (-{n_before - n_no_nan:,})",
    f"    ├─ Filtrar created_at <= REF           (-{n_no_nan - n_created_ok:,})",
    f"    ├─ Mapear player_id (char_id) → user_id usando characters.csv",
    f"    │   ({len(char_to_user):,} char_id → user_id, descartar huérfanos -{n_unmap_filter:,})",
    f"    └─ Filtrar por sample_user_ids         (-{n_mapped - n_in_sample:,})",
    "",
    f"Filtrado ({n_in_sample:,} filas)",
    "    │",
    "    ├─ Grupo A: volumen (3)",
    "    ├─ Grupo B: performance (3)",
    "    ├─ Grupo C: diversidad (1)",
    "    ├─ Grupo D: temporal (3)",
    "    ├─ Grupo E: engagement (2)",
    "    └─ Reindex con sample_user_ids + fillna",
    "",
    f"features_fights_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero (genera sample_user_ids)",
    "2. (Opcional) Ejecutar 02c, 02e, 02i para los cross-checks",
    "3. Ejecutar 02j_fights_log.ipynb",
    "4. Verificar: features_fights_cutoff.parquet == 126.253 filas × 13 cols",
    "",
    "---", "",
    "## Referencias",
    "",
    "- PLANTILLA_NOTEBOOK_02.md — estructura estándar de notebook Fase 1",
    "- 02g_devices.ipynb — referencia para parser epoch+Timedelta",
    "- 02f_iaps.ipynb — referencia para sanity checks point-in-time",
    "- Hadiji et al. (2014). *Predicting player churn in the wild* — relación",
    "  PvP win rate ↔ churn",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')

In [ ]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02j_fights_log")
print("=" * 70)
print(f"  Tiempo total          : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV (con usecols): {n_before:,}")
print(f"  Filas filtradas       : {n_in_sample:,} ({n_in_sample/n_before*100:.1f}%)")
print(f"  Usuarios con combates : {int((features['fights_total']>0).sum()):,} ({int((features['fights_total']>0).sum())/len(features)*100:.1f}%)")
print(f"  fights_total mediana  : {features['fights_total'].median():.0f}")
print(f"  fights_total max      : {features['fights_total'].max():,}")
print(f"  fights_pvp_total > 0  : {int((features['fights_pvp_total']>0).sum()):,}")
print(f"  fights_win_rate p50   : {features[features['fights_total']>0]['fights_win_rate'].median():.3f}")
print(f"  fights_pvp_win_rate p50: {features[features['fights_pvp_total']>0]['fights_pvp_win_rate'].median():.3f}")
print(f"  Filas features final  : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features     : {features.shape[1]}")
print()
print("Archivos generados:")
print(f"  features_fights_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.1f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)
print("Siguiente paso: revisar execution_report.md y compartirlo con Claude")

In [24]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02j_fights_log.ipynb'
html_path = REPORT_DIR / '02j_fights_log_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(n_before, len(USECOLS)),
    filter_steps=[
        ('CSV con usecols', n_before),
        ('Sin player_id NaN', n_no_nan),
        ('created_at <= REF', n_created_ok),
        ('Tras mapeo char→user', n_mapped),
        ('En sample', n_in_sample),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")

HTML generado: 02j_fights_log_run.html (0.5 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/fights_log/execution_report.md
